In [1]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader
from openpi_client import image_tools

import Pyro5.api
import numpy as np

In [ ]:
# config = _config.get_config("pi0_fast_droid")
# checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_fast_droid")
# print(f"Loading model from {checkpoint_dir}")

In [2]:
# CHANGE 'action' to 'actions' IN NORM_STATS.JSON FIRST
%env CUDA_VISIBLE_DEVICES=6,7
# load the model
config = _config.get_config("pi0_RPM_low_mem_finetune")
# checkpoint_dir = '/data/checkpoints/pi0_RPM_low_mem_finetune/LoRA_2GPU_pickblueblockblackbowl/4000'
checkpoint_dir = '/home/liao0241/openpi/checkpoints/pi0_RPM_low_mem_finetune/LoRA_2GPU_pickblueblockblackbowl_presumablycorrectednormstats/7000'

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

env: CUDA_VISIBLE_DEVICES=6,7


In [ ]:
np.array(
    [-3.018277883529663, -2.4735472202301025, -1.4247097969055176, -1.420408844947815, 0.47022080421447754, 0.31647270917892456]
    + [-0.6982309818267822, 0.13667820394039154, 0.06255360692739487]
    + [-1.4045917987823486, -1.7060638666152954, -1.2871901988983154]
    + [3]  # gripper
)

In [ ]:
# delete current policy to free up memory, then load a new one
# del policy

In [ ]:
# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
# example = droid_policy.make_droid_example()
# result = policy.infer(example)
# print("Actions shape:", result["actions"].shape)

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2
import matplotlib.pyplot as plt

class RealSenseCamera:
    def __init__(self, serial_number, width=640, height=480, fps=30):
        self.serial = serial_number
        self.pipeline = rs.pipeline()
        self.config = rs.config()
        self.config.enable_device(self.serial)
        self.config.enable_stream(rs.stream.color, width, height, rs.format.bgr8, fps)
        self.pipeline.start(self.config)

    def get_image(self):
        frames = self.pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame:
            return None

        bgr = np.asanyarray(color_frame.get_data())
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        return rgb

    def release(self):
        self.pipeline.stop()


# initialize cameras
serial_head = "f1380660"
serial_wrist = "128422270284"
cam_head = RealSenseCamera(serial_head)
cam_wrist = RealSenseCamera(serial_wrist)

RuntimeError: No device connected

In [ ]:
from openpi_client import image_tools

img_head = image_tools.resize_with_pad(
                        cam_head.get_image(), 224, 224
                    )

img_wrist = image_tools.resize_with_pad(
                        cam_wrist.get_image(), 224, 224
                    )


# combine the two images
combined = np.hstack((img_head, img_wrist))  # shape (224, 224*2, 3)

# display the combined image
plt.figure(figsize=(8, 4))
plt.imshow(combined)
plt.title("Head + Wrist(RGB 224x224)")
plt.axis('off')
plt.show()

In [ ]:
cam_head.release()
cam_wrist.release()

In [ ]:
# Test: feed dummy data

import cv2
import numpy as np
# Assuming 'image_tools' is available in your environment for resizing
# import image_tools 

# Getting first frame from observation.image_scene and observation.image_wrist videos
video_scene = '/home/liao0241/.cache/huggingface/lerobot/iamandrewliao/pickblueblock_blackbowl/videos/chunk-000/observation.image_scene/episode_000000.mp4'
video_wrist = '/home/liao0241/.cache/huggingface/lerobot/iamandrewliao/pickblueblock_blackbowl/videos/chunk-000/observation.image_wrist/episode_000000.mp4'

# Initialize video capture objects
cap_head = cv2.VideoCapture(video_scene)
cap_wrist = cv2.VideoCapture(video_wrist)

# Basic check to ensure videos opened
if not cap_head.isOpened():
    raise IOError(f"Error opening head camera video: {video_scene}")
if not cap_wrist.isOpened():
    raise IOError(f"Error opening wrist camera video: {video_wrist}")

# Read the next frame from the videos
ret_head, frame_head = cap_head.read()
ret_wrist, frame_wrist = cap_wrist.read()

# Check if both frames were successfully read
if not ret_head or not ret_wrist:
    # Handle the end of video or read error (e.g., break a loop, print an error)
    print("Warning: Could not read one or both frames. Ending process.")
    # In a real application, you'd break the loop or seek/reset the video.
    # For this example, we'll stop execution if a frame is missing.
    # IMPORTANT: You MUST release the capture objects when done (see cleanup below)
    cap_head.release()
    cap_wrist.release()
    # raise StopIteration("One or both videos have ended.")
    # If this is inside a loop, use 'continue' or 'break' as appropriate.
    pass # Placeholder for error handling

# Ensure frames are in the required format (RGB) for policy inference
# OpenCV reads frames in BGR format by default.
frame_head_rgb = cv2.cvtColor(frame_head, cv2.COLOR_BGR2RGB)
frame_wrist_rgb = cv2.cvtColor(frame_wrist, cv2.COLOR_BGR2RGB)

# Resize frames
# Alternatively, you can use cv2.resize for a basic resize:
# img_scene = cv2.resize(frame_head_rgb, (224, 224))
# img_wrist = cv2.resize(frame_wrist_rgb, (224, 224))
# img_scene = image_tools.resize_with_pad(frame_head_rgb, 224, 224)
# img_wrist = image_tools.resize_with_pad(frame_wrist_rgb, 224, 224)

# First observation.state in a demo
dummy_state = np.array(
    [-3.018277883529663, -2.4735472202301025, -1.4247097969055176, -1.420408844947815, 0.47022080421447754, 0.31647270917892456]
    + [-0.6982309818267822, 0.13667820394039154, 0.06255360692739487]
    + [-1.4045917987823486, -1.7060638666152954, -1.2871901988983154]
    + [3]  # gripper
)

print("State: ", dummy_state)

observation = {
    # 'observation.image_scene': img_scene,
    # # 'observation.image_wrist': img_wrist,
    'observation.image_scene': frame_head_rgb,
    'observation.image_wrist': frame_wrist_rgb,
    # "observation.state": np.array([0]*13, dtype=np.float32),
    "observation.state": dummy_state.astype(np.float32),
    # "observation/gripper_position": np.array([0], dtype=np.float32),  
    # "observation/joint_position": np.zeros_like(obs["robot_state"], dtype=np.float32),
    # "observation/gripper_position": np.zeros_like([obs["gripper_state"]], dtype=np.float32),  
    "prompt": "pick the blue block and place it in the black bowl",
}

result = policy.infer(observation) 

action_list = result["actions"][0]

print("Predicted actions: ", action_list)

# --- Cleanup: Release Video Capture Objects ---
# IMPORTANT: This block should be executed once all frame reading is complete, 
# typically after a loop or at the end of your script.
cap_head.release()
cap_wrist.release()

State:  [-3.01827788 -2.47354722 -1.4247098  -1.42040884  0.4702208   0.31647271
 -0.69823098  0.1366782   0.06255361 -1.4045918  -1.70606387 -1.2871902
  3.        ]
Predicted actions:  [-3.02497051e+00 -2.47922814e+00 -1.41570599e+00 -1.42184871e+00
  4.64949103e-01  3.03572593e-01 -6.05247101e-04]


In [ ]:
# Get ground truth actions for comparison
import pickle

def inspect_pickle_file(pickle_file_path):
    """Extract and display the contents of a pickle file."""
    try:
        with open(pickle_file_path, 'rb') as f:
            data = pickle.load(f)
            
        # print(f"Contents of {pickle_file_path}:")
        
        # If it's a dictionary, display the keys and values
        if isinstance(data, dict):
            # print(data['meta'])
            for key, value in data.items():
                # print(type(value))
                if isinstance(value, list):
                    # print(f"- {key}: List of length {len(value)}")
                    for timestep in value:
                        # print(timestep.keys())
                        # print(timestep['joint_positions'])
                        # print(timestep['eef_pose']['position'], timestep['eef_pose']['orientation_rpy'])
                        # print(timestep['gripper_state'])
                        print(timestep['spark_command_angles'])
                        print(timestep['spark_command_gripper'])
                        break
                # print(f"- {key}: {value}")
        else:
            print(data)
            
    except Exception as e:
        print(f"Error opening {pickle_file_path}: {e}")

print("State: ", dummy_state)
print("Predicted actions: ", action_list)

GT_path = '/data/shared_data/real_world_data/pickblueblock_blackbowl/traj_1758307262.pkl'
print("Ground Truth Actions: ")
print(inspect_pickle_file(GT_path))

State:  [-3.01827788 -2.47354722 -1.4247098  -1.42040884  0.4702208   0.31647271
 -0.69823098  0.1366782   0.06255361 -1.4045918  -1.70606387 -1.2871902
  3.        ]
Predicted actions:  [-3.02497051e+00 -2.47922814e+00 -1.41570599e+00 -1.42184871e+00
  4.64949103e-01  3.03572593e-01 -6.05247101e-04]
Ground Truth Actions: 
array('f', [-3.018874168395996, -2.47239351272583, -1.4243011474609375, -1.420082688331604, 0.4705485999584198, 0.31676700711250305])
0.0
None


In [ ]:
# Actual deployment
# vla_policy_client
import Pyro5.api
import numpy as np
import time
from openpi_client import image_tools

ns = Pyro5.api.locate_ns()  # Locate the name server
uri = ns.lookup("pi0_controller")  # Use the name server to look up the URI
controller = Pyro5.api.Proxy(uri)

# prompt
prompt = "pick up the blue block and place it into the black bowl"  # <-- change prompt

MAX_STEPS = 50000  # maximum number of steps to run
MAX_NORM = 0.8  # change as appropriate
video_buffer = []  # buffer to store video frames

# # dummy action
# action = np.zeros((50,7), dtype=np.float32)
# action_list = action.tolist()  # convert to list for sending

for step in range(MAX_STEPS):
    start = time.time()
    print(f"\n=== Step {step} ===")
    # Send action to controller to execute
    data_to_send = {
        "type": "action",
        "data": action_list,
        "step": step
    }
    start1 = time.time()
    print(f"Sending data to controller: {data_to_send}")
    obs = controller.step(data_to_send) # returns {'robot state':...,'step':...}
    end1 = time.time()
    print(f"Controller step took {end1 - start1:.2f} seconds")


    # Create an observation to do the next inference
    img_left = image_tools.resize_with_pad(
                            cam_head.get_image(), 224, 224
                        )
    img_wrist = image_tools.resize_with_pad(
                            cam_wrist.get_image(), 224, 224
                        )
    # save the images
    combined = np.hstack([img_left, img_wrist])
    video_buffer.append(combined)

    observation = {
        "observation.image_scene": img_left,
        # "observation/exterior_image_1_left": np.zeros_like(img_left),  # dummy image for left
        "observation.image_wrist": img_wrist,
        # "observation/wrist_image_left": np.zeros_like(img_wrist),  # dummy image for wrist
        "observation.state": np.array(obs["robot_state"], dtype=np.float32),
        # "observation/gripper_position": np.array([obs["gripper_state"]], dtype=np.float32),  
        # "observation/joint_position": np.zeros_like(obs["robot_state"], dtype=np.float32),
        # "observation/gripper_position": np.zeros_like([obs["gripper_state"]], dtype=np.float32),  
        "prompt": prompt,
    }


    # Inference
    result = policy.infer(observation)

    curr_action = result["actions"]
    # Clip actions if norm of delta is too high
    delta_action = curr_action - observation['observation.state']
    def clip_action(action, max_norm):
        norm = np.lingalg.norm(action)
        if norm > max_norm:
            # Scale down the vector to the max_norm while preserving its direction
            clipped_action = (action / norm) * max_norm
            print(f"Action clipped. Original norm: {norm:.4f}, New norm: {max_norm:.4f}")
            return clipped_action
        else:
            # If the norm is within the threshold, return the original action
            print(f"Action not clipped. Norm: {norm:.4f}")
            return action
    final_action = clip_action(delta_action, MAX_NORM)

    action_list = final_action.tolist()
    
    end2 = time.time()
    print(f"Inference took {end2 - end1:.2f} seconds")
    print(f"Action at step {step}:", action_list[0])
    end = time.time()
    print(f"Step {step} took {end - start:.2f} seconds")